# ConFit trên resume-job-fit (cnamuangtoun) – GPU

**Settings → Accelerator: GPU T4 x2 hoặc P100**, **Internet: On** (để clone repo và tải dữ liệu từ HuggingFace). Không cần khóa API.

Bấm **Save Version → Save & Run All**. Xong thì tải `confit_ext.csv` và `confit_ext_val.csv` ở tab Output, đặt vào `report/results/` trên máy rồi chạy `python report/b_eval_ext.py --tag _jd10 --main pair_skip_mlp_specfeat,pair_skip_ctx_spec`.

File được ghi sau mỗi fold, nên nếu phiên dừng giữa chừng vẫn tải được phần đã chạy.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!git clone -q -b bang-branch https://github.com/hoangphungvan/doantotnghiep.git
%cd doantotnghiep
!git log --oneline -1

In [ ]:
# Chỉ cài phần còn thiếu, KHÔNG cài lại torch (giữ bản CUDA của Kaggle)
!pip install -q torch_geometric python-dotenv openai pymupdf
import torch, sentence_transformers; print(torch.__version__, torch.cuda.is_available(), sentence_transformers.__version__)

In [ ]:
# Bộ dữ liệu không nằm trong repo: tải train.csv/test.csv gốc từ HuggingFace
from huggingface_hub import snapshot_download
snapshot_download('cnamuangtoun/resume-job-description-fit', repo_type='dataset', local_dir='data/raw/cnamuangtoun')
!ls -la data/raw/cnamuangtoun

In [ ]:
# 5 seed x 5 fold theo JD. Dòng đầu phải in '7993 cặp' (khớp tập cặp của mô hình chính)
!mkdir -p /kaggle/working
!python -X utf8 src/modeling/train_confit.py ext --seeds 5 --maxlen 512 --epochs 3 && cp report/results/confit_ext*.csv /kaggle/working/

In [ ]:
!cp report/results/confit_ext*.csv /kaggle/working/ 2>/dev/null; ls -la /kaggle/working/*.csv